In [2]:
# Install required libraries
!pip install google-api-python-client
!pip install pandas
!pip install python-dotenv
!pip install mysql-connector-python

In [2]:
# Imports
import os
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
import mysql.connector as mc

# Load environment variables
load_dotenv()

# YouTube API
api_key = os.getenv("API_KEY")
connection = build('youtube', 'v3', developerKey=api_key)

# MySQL connection
mydb = mc.connect(
    host=os.getenv("HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("PASSWORD"),
    database=os.getenv("DATABASE")
)

cursor = mydb.cursor(buffered=True)

In [7]:
# ------------------ CHANNEL ------------------
def fetch_channel_info(channel_id):
    response = connection.channels().list(
        part="snippet,statistics,contentDetails",
        id=channel_id
    ).execute()

    item = response['items'][0]

    return {
        "channel_id": item['id'],
        "title": item['snippet']['title'],
        "description": item['snippet']['description'],
        "subscriber_count": item['statistics']['subscriberCount'],
        "view_count": item['statistics']['viewCount'],
        "video_count": item['statistics']['videoCount'],
        "playlist_id": item['contentDetails']['relatedPlaylists']['uploads']
    }

# ------------------ VIDEO IDS ------------------
def fetch_video_ids(playlist_id):
    video_ids = []

    request = connection.playlistItems().list(
        part="contentDetails",
        playlistId=playlist_id
    ).execute()

    while True:
        for item in request['items']:
            video_ids.append(item['contentDetails']['videoId'])

        token = request.get('nextPageToken')
        if not token:
            break

        request = connection.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            pageToken=token
        ).execute()

    return video_ids

# ------------------ VIDEO DETAILS ------------------
def fetch_video_details(video_id):
    response = connection.videos().list(
        part="snippet,statistics,contentDetails",
        id=video_id
    ).execute()

    item = response.get('items', [{}])[0]

    snippet = item.get('snippet', {})
    stats = item.get('statistics', {})
    details = item.get('contentDetails', {})

    return {
        "video_id": item.get('id'),
        "title": snippet.get('title', ""),
        "description": snippet.get('description', ""),
        "likes": stats.get('likeCount', 0),
        "views": stats.get('viewCount', 0),
        "comment_count": stats.get('commentCount', 0),
        "duration": details.get('duration', ""),
        "published_at": snippet.get('publishedAt', "").replace('T', ' ').replace('Z', '')
    }

# ------------------ COMMENTS ------------------
def fetch_comments(video_id):
    comments = []

    try:
        request = connection.commentThreads().list(
            part="snippet",
            videoId=video_id
        ).execute()

        while True:
            for item in request.get('items', []):
                snippet = item['snippet']['topLevelComment']['snippet']

                comments.append({
                    "comment_id": item['snippet']['topLevelComment']['id'],
                    "comment": snippet.get('textDisplay', "")
                })

            token = request.get('nextPageToken')
            if not token:
                break

            request = connection.commentThreads().list(
                part="snippet",
                videoId=video_id,
                pageToken=token
            ).execute()

    except Exception as e:
        print("Comment fetch error:", e)

    return comments

# ------------------ MASTER FETCH ------------------
def fetch_all_video_data(channel_id):
    channel_info = fetch_channel_info(channel_id)
    video_ids = fetch_video_ids(channel_info['playlist_id'])

    videos = []

    for vid in video_ids:
        videos.append({
            "video_details": fetch_video_details(vid),
            "comments": fetch_comments(vid)
        })

    return {
        "channel_info": channel_info,
        "videos": videos
    }

In [10]:
# ------------------ CHANNEL ------------------
def insert_channel_data(channel_info):
    cursor.execute("""
    INSERT IGNORE INTO channels VALUES (%s,%s,%s,%s,%s,%s,%s)
    """, (
        channel_info['channel_id'],
        channel_info['title'],
        channel_info['description'],
        int(channel_info['subscriber_count']),
        int(channel_info['view_count']),
        int(channel_info['video_count']),
        channel_info['playlist_id']
    ))
    mydb.commit()

# ------------------ VIDEO ------------------
def insert_video_data(video_info, channel_id):
    cursor.execute("""
    INSERT IGNORE INTO videos VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        video_info['video_id'],
        channel_id,
        video_info['title'],
        video_info['description'],
        int(video_info['likes']),
        int(video_info['views']),
        int(video_info['comment_count']),
        video_info['duration'],
        video_info['published_at']
    ))
    mydb.commit()

# ------------------ COMMENTS ------------------
def insert_comment_data(comment_info, video_id):
    cursor.execute("""
    INSERT IGNORE INTO comments VALUES (%s,%s,%s)
    """, (
        comment_info['comment_id'],
        video_id,
        comment_info['comment']
    ))
    mydb.commit()

# ------------------ PIPELINE ------------------
def store_channel_data(channel_id):
    channel_info = fetch_channel_info(channel_id)
    insert_channel_data(channel_info)

    video_ids = fetch_video_ids(channel_info['playlist_id'])

    for vid in video_ids:
        video_info = fetch_video_details(vid)
        insert_video_data(video_info, channel_id)

        comments = fetch_comments(vid)
        for c in comments:
            insert_comment_data(c, vid)

In [11]:
cursor.execute("DROP TABLE IF EXISTS comments")
cursor.execute("DROP TABLE IF EXISTS videos")
cursor.execute("DROP TABLE IF EXISTS channels")

cursor.execute("""
CREATE TABLE channels (
    channel_ID VARCHAR(50) PRIMARY KEY,
    title VARCHAR(255),
    description TEXT,
    subsciber_count BIGINT,
    view_count BIGINT,
    video_count BIGINT,
    playlist_id VARCHAR(50)
)
""")

cursor.execute("""
CREATE TABLE videos (
    video_ID VARCHAR(50) PRIMARY KEY,
    channel_ID VARCHAR(50),
    title VARCHAR(255),
    description TEXT,
    likes BIGINT,
    view_count BIGINT,
    comment_count BIGINT,
    duration VARCHAR(50),
    published_at TIMESTAMP,
    FOREIGN KEY (channel_ID) REFERENCES channels(channel_ID)
)
""")

cursor.execute("""
CREATE TABLE comments (
    comment_ID VARCHAR(100) PRIMARY KEY,
    video_ID VARCHAR(50),
    comment_text TEXT,
    FOREIGN KEY (video_ID) REFERENCES videos(video_ID)
)
""")

print("✅ Tables created")

✅ Tables created


In [9]:
channel_id = 'UCxpNh5gDujzK_Bn19tF-ASg'

store_channel_data(channel_id)

# Verify
cursor.execute("SELECT COUNT(*) FROM videos")
print("Videos:", cursor.fetchone())

cursor.execute("SELECT COUNT(*) FROM comments")
print("Comments:", cursor.fetchone())

Videos: (33,)
Comments: (535,)


In [12]:
channel_id = 'UCxpNh5gDujzK_Bn19tF-ASg'

store_channel_data(channel_id)

# Verify
cursor.execute("SELECT COUNT(*) FROM videos")
print("Videos:", cursor.fetchone())

cursor.execute("SELECT COUNT(*) FROM comments")
print("Comments:", cursor.fetchone())

Videos: (33,)
Comments: (535,)


In [13]:
fetch_comments('p5DRcllp0nQ')

[{'comment_id': 'UgzqCVIo_RZ9EUO0a3t4AaABAg', 'comment': 'Wow😍'},
 {'comment_id': 'UgydE3YmPKfbRPBMNlZ4AaABAg', 'comment': 'Best'},
 {'comment_id': 'UgwyFymMH6_RUfeOqiR4AaABAg', 'comment': 'Keep going bro'},
 {'comment_id': 'UgzYtmbTGqhs35LcXEx4AaABAg',
  'comment': 'nikhilesh bhai mere apise de de mai anshul'},
 {'comment_id': 'Ugiomau6aKHa43gCoAEC',
  'comment': 'Great work guys! Nikhilesh, nice work with the classical stuff. Keep it going!'},
 {'comment_id': 'Ughoy0PBkE_AP3gCoAEC',
  'comment': 'Awesome work..Watch this cover too<br>MUST WATCH - Judaa Unplugged (Magical)<br><a href="https://www.youtube.com/watch?v=B7LCCJ5Wcok">https://youtu.be/B7LCCJ5Wcok</a>'}]

In [137]:
def fetch_all_video_data(channel_id):

    channel_info = fetch_channel_info(channel_id)
    playlist_id = channel_info['playlist_id']

    video_ids = fetch_video_ids(playlist_id)

    all_videos = []

    for video_id in video_ids:
        video_details = fetch_video_details(video_id)
        comments = fetch_comments(video_id)

        all_videos.append({
            "video_id": video_id,
            "video_details": video_details,
            "comments": comments
        })

    return {
        "channel_info": channel_info,
        "videos": all_videos
    }

In [3]:
cursor.execute("SELECT * FROM channels")
print(cursor.fetchall())

cursor.execute("SELECT * FROM videos LIMIT 5")
print(cursor.fetchall())

cursor.execute("SELECT * FROM comments LIMIT 5")
print(cursor.fetchall())

[('UCxpNh5gDujzK_Bn19tF-ASg', 'Akash Ravuru', 'https://www.instagram.com/akashravuru/', 2040, 320315, 33, 'UUxpNh5gDujzK_Bn19tF-ASg')]
[('3iwh0NMVIq4', 'UCxpNh5gDujzK_Bn19tF-ASg', '#ArereyManasa | Falaknuma Das | Cover | Vishwak Sen, #SidSriram Vivek Sagar |  Akash Ravuru |', 'I was just randomly humming the song and my mom shot a little video of it. Put up a story but A lot of you wanted me to post it as a reel. So yay 👀\n.\n.\nAlso this is super raw and random.\nHope you like it. Peace! ✌🏻\n.\n\n#falaknamadas #sidsriram #viveksagar #vishwaksen\n#SidSriramMelodyHits #SidSriramMelodySongsCollection #FalaknamaDas #masskadas #Vishwaksen #TeluguCovers #RawTelugusongs #Tollywood #AcousticCovers\n.\n.', 500, 8082, 16, 'PT46S', datetime.datetime(2022, 1, 6, 6, 16, 17)), ('4_xncWNc_xc', 'UCxpNh5gDujzK_Bn19tF-ASg', 'Evarevaro x Pehle Bhi Main | Animal | Akash Ravuru | Cover | Ranbir Kapoor | Rashmika Mandanna|', 'Emo. Em chestunnano, Inka em em chesthano. Chesthuu em aypothano Mari?\n-\n-\n-\n

In [8]:
###QUESTION 1
cursor.execute("""SELECT videos.title, channels.title 
FROM videos
JOIN channels ON videos.channel_ID = channels.channel_ID""")
print(cursor.fetchall())

[('#ArereyManasa | Falaknuma Das | Cover | Vishwak Sen, #SidSriram Vivek Sagar |  Akash Ravuru |', 'Akash Ravuru'), ('Evarevaro x Pehle Bhi Main | Animal | Akash Ravuru | Cover | Ranbir Kapoor | Rashmika Mandanna|', 'Akash Ravuru'), ('#manohara × #zarazara  | RHTDM | Acoustic Cover | Akash Ravuru | #Madhavan', 'Akash Ravuru'), ('O Rendu Prema Meghaalila | Baby | Akash Ravuru | Cover | Anand Devarkonda | Vaishnavi Chaitanya', 'Akash Ravuru'), ('Inthandham | Sita Ramam (Telugu) | Dulquer | Mrunal | Vishal | Akash Ravuru | Cover |', 'Akash Ravuru'), ('Kabira/Yaariyaan MASHUP by Akash Ravuru & Sidharth Bendi', 'Akash Ravuru'), ('O Re Piya Song | Cover | Aaja Nachle | Akash Ravuru | Rahat Fateh Ali Khan |', 'Akash Ravuru'), ('Kannulo Nee Roopame - Ninne Pelladtha | Cover | Akash Ravuru', 'Akash Ravuru'), ('Prashna | Akash Ravuru | Original #music #explorepage #telugu #bollywood #originalmusic #song', 'Akash Ravuru'), ('Kesariya x Kumkumala | Bramhastra | Arijit Singh  | Sid Sriram | Pritam 

Question 2 

In [9]:

cursor.execute("""
SELECT channels.title, COUNT(videos.video_ID) AS video_count
FROM videos
JOIN channels ON videos.channel_ID = channels.channel_ID
GROUP BY channels.channel_ID
ORDER BY video_count DESC
""")

print(cursor.fetchall())

[('Akash Ravuru', 33)]


Question 3 

In [10]:
cursor.execute("""
SELECT videos.title, channels.title, videos.view_count
FROM videos
JOIN channels ON videos.channel_ID = channels.channel_ID
ORDER BY videos.view_count DESC
LIMIT 10
""")

print(cursor.fetchall())

[('Srivalli | Pushpa-The Rise |Acoustic Cover|Akash Ravuru|#AlluArjun, #Rashmika | #DSP | #SidSriram', 'Akash Ravuru', 119856), ('O Cheliya | Premikudu | Cover | Ar Rahman | Akash Ravuru', 'Akash Ravuru', 47248), ('Ye Chilipi | Gharshana | Venkatesh | Asin |  Akash Ravuru | Cover |', 'Akash Ravuru', 25104), ('Inthandham | Sita Ramam (Telugu) | Dulquer | Mrunal | Vishal | Akash Ravuru | Cover |', 'Akash Ravuru', 24811), ('#ManasaVacha | Acoustic Cover | Sumanth | Kamalini Mukharjee || Akash Ravuru | #Godavari', 'Akash Ravuru', 12361), ('Kesariya x Kumkumala | Bramhastra | Arijit Singh  | Sid Sriram | Pritam | Cover | Akash Ravuru', 'Akash Ravuru', 9413), ('#Sirivennela  | Shyam Singha Roy | Cover | #Nani #SaiPallavi | Akash Ravuru', 'Akash Ravuru', 8624), ('#ArereyManasa | Falaknuma Das | Cover | Vishwak Sen, #SidSriram Vivek Sagar |  Akash Ravuru |', 'Akash Ravuru', 8082), ('Judaa -Ishqderiyaan (Acoustic Guitar Cover). Nikhilesh Garnepudy ft Akash Ravuru.', 'Akash Ravuru', 7084), ('Kab

Question 4

In [11]:
cursor.execute("""
SELECT videos.title, COUNT(comments.comment_ID) AS comment_count
FROM videos
LEFT JOIN comments 
ON videos.video_ID = comments.video_ID
GROUP BY videos.video_ID
ORDER BY comment_count DESC
""")

print(cursor.fetchall())

[('Srivalli | Pushpa-The Rise |Acoustic Cover|Akash Ravuru|#AlluArjun, #Rashmika | #DSP | #SidSriram', 136), ('O Cheliya | Premikudu | Cover | Ar Rahman | Akash Ravuru', 67), ('Ye Chilipi | Gharshana | Venkatesh | Asin |  Akash Ravuru | Cover |', 44), ('Inthandham | Sita Ramam (Telugu) | Dulquer | Mrunal | Vishal | Akash Ravuru | Cover |', 33), ('#ManasaVacha | Acoustic Cover | Sumanth | Kamalini Mukharjee || Akash Ravuru | #Godavari', 20), ('Kesariya x Kumkumala | Bramhastra | Arijit Singh  | Sid Sriram | Pritam | Cover | Akash Ravuru', 18), ('#Sirivennela  | Shyam Singha Roy | Cover | #Nani #SaiPallavi | Akash Ravuru', 17), ('A R Rahman | Chakori | Cover |  Akash Ravuru Ft. Nikhil Valaboju', 15), ('Ye Cheekati | Vedam | #AlluArjun | #ManchuManoj | #AnushkaShetty | #MMKeeravani | Akash Ravuru |', 14), ('Appudo Ippudo | Bommarillu | Akash Ravuru | Cover | DSP | Siddharth | Genelia', 13), ('#NewYorkNagaram | Nuvvu Nenu Prema  | #Suriya, #Jyothika | #ArRahman | Akash Ravuru', 12), ('#und

Question 5


In [12]:
cursor.execute("""
SELECT videos.title, channels.title, videos.likes
FROM videos
JOIN channels ON videos.channel_ID = channels.channel_ID
ORDER BY videos.likes DESC
LIMIT 10
""")

print(cursor.fetchall())

[('Srivalli | Pushpa-The Rise |Acoustic Cover|Akash Ravuru|#AlluArjun, #Rashmika | #DSP | #SidSriram', 'Akash Ravuru', 7992), ('O Cheliya | Premikudu | Cover | Ar Rahman | Akash Ravuru', 'Akash Ravuru', 3039), ('Ye Chilipi | Gharshana | Venkatesh | Asin |  Akash Ravuru | Cover |', 'Akash Ravuru', 1553), ('Inthandham | Sita Ramam (Telugu) | Dulquer | Mrunal | Vishal | Akash Ravuru | Cover |', 'Akash Ravuru', 1014), ('#ManasaVacha | Acoustic Cover | Sumanth | Kamalini Mukharjee || Akash Ravuru | #Godavari', 'Akash Ravuru', 687), ('Kesariya x Kumkumala | Bramhastra | Arijit Singh  | Sid Sriram | Pritam | Cover | Akash Ravuru', 'Akash Ravuru', 653), ('#ArereyManasa | Falaknuma Das | Cover | Vishwak Sen, #SidSriram Vivek Sagar |  Akash Ravuru |', 'Akash Ravuru', 500), ('#Sirivennela  | Shyam Singha Roy | Cover | #Nani #SaiPallavi | Akash Ravuru', 'Akash Ravuru', 492), ('Evare | #Premam | Acoustic Cover |#NagaChaitanya | #ShruthiHassan | #Anupama | Akash Ravuru |', 'Akash Ravuru', 388), ('O 

Question 6

In [14]:
cursor.execute("""
SELECT title, likes, view_count
FROM videos
ORDER BY likes DESC
LIMIT 10
""")
print(cursor.fetchall())

[('Srivalli | Pushpa-The Rise |Acoustic Cover|Akash Ravuru|#AlluArjun, #Rashmika | #DSP | #SidSriram', 7992, 119856), ('O Cheliya | Premikudu | Cover | Ar Rahman | Akash Ravuru', 3039, 47248), ('Ye Chilipi | Gharshana | Venkatesh | Asin |  Akash Ravuru | Cover |', 1553, 25104), ('Inthandham | Sita Ramam (Telugu) | Dulquer | Mrunal | Vishal | Akash Ravuru | Cover |', 1014, 24811), ('#ManasaVacha | Acoustic Cover | Sumanth | Kamalini Mukharjee || Akash Ravuru | #Godavari', 687, 12361), ('Kesariya x Kumkumala | Bramhastra | Arijit Singh  | Sid Sriram | Pritam | Cover | Akash Ravuru', 653, 9413), ('#ArereyManasa | Falaknuma Das | Cover | Vishwak Sen, #SidSriram Vivek Sagar |  Akash Ravuru |', 500, 8082), ('#Sirivennela  | Shyam Singha Roy | Cover | #Nani #SaiPallavi | Akash Ravuru', 492, 8624), ('Evare | #Premam | Acoustic Cover |#NagaChaitanya | #ShruthiHassan | #Anupama | Akash Ravuru |', 388, 5970), ('O Rendu Prema Meghaalila | Baby | Akash Ravuru | Cover | Anand Devarkonda | Vaishnavi 

Question 7

In [15]:
cursor.execute("""
SELECT title, view_count
FROM channels
ORDER BY view_count DESC
""")

print(cursor.fetchall())

[('Akash Ravuru', 320315)]


Question 8 

In [16]:
cursor.execute("""
SELECT DISTINCT channels.title
FROM videos
JOIN channels ON videos.channel_ID = channels.channel_ID
WHERE YEAR(videos.published_at) = 2022
""")

print(cursor.fetchall())

[('Akash Ravuru',)]


Question 9

In [18]:
cursor.execute("""
SELECT channels.title, videos.title, videos.duration
FROM videos
JOIN channels ON videos.channel_ID = channels.channel_ID
ORDER BY channels.title
""")

print(cursor.fetchall())

[('Akash Ravuru', '#ArereyManasa | Falaknuma Das | Cover | Vishwak Sen, #SidSriram Vivek Sagar |  Akash Ravuru |', 'PT46S'), ('Akash Ravuru', 'Evarevaro x Pehle Bhi Main | Animal | Akash Ravuru | Cover | Ranbir Kapoor | Rashmika Mandanna|', 'PT1M5S'), ('Akash Ravuru', '#manohara × #zarazara  | RHTDM | Acoustic Cover | Akash Ravuru | #Madhavan', 'PT1M53S'), ('Akash Ravuru', 'O Rendu Prema Meghaalila | Baby | Akash Ravuru | Cover | Anand Devarkonda | Vaishnavi Chaitanya', 'PT1M18S'), ('Akash Ravuru', 'Inthandham | Sita Ramam (Telugu) | Dulquer | Mrunal | Vishal | Akash Ravuru | Cover |', 'PT1M32S'), ('Akash Ravuru', 'Kabira/Yaariyaan MASHUP by Akash Ravuru & Sidharth Bendi', 'PT5M23S'), ('Akash Ravuru', 'O Re Piya Song | Cover | Aaja Nachle | Akash Ravuru | Rahat Fateh Ali Khan |', 'PT1M11S'), ('Akash Ravuru', 'Kannulo Nee Roopame - Ninne Pelladtha | Cover | Akash Ravuru', 'PT1M28S'), ('Akash Ravuru', 'Prashna | Akash Ravuru | Original #music #explorepage #telugu #bollywood #originalmusi

Question 10

In [17]:
cursor.execute("""
SELECT videos.title, channels.title, videos.comment_count
FROM videos
JOIN channels ON videos.channel_ID = channels.channel_ID
ORDER BY videos.comment_count DESC
LIMIT 10
""")

print(cursor.fetchall())

[('Srivalli | Pushpa-The Rise |Acoustic Cover|Akash Ravuru|#AlluArjun, #Rashmika | #DSP | #SidSriram', 'Akash Ravuru', 263), ('O Cheliya | Premikudu | Cover | Ar Rahman | Akash Ravuru', 'Akash Ravuru', 83), ('Inthandham | Sita Ramam (Telugu) | Dulquer | Mrunal | Vishal | Akash Ravuru | Cover |', 'Akash Ravuru', 64), ('Ye Chilipi | Gharshana | Venkatesh | Asin |  Akash Ravuru | Cover |', 'Akash Ravuru', 50), ('A R Rahman | Chakori | Cover |  Akash Ravuru Ft. Nikhil Valaboju', 'Akash Ravuru', 29), ('#ManasaVacha | Acoustic Cover | Sumanth | Kamalini Mukharjee || Akash Ravuru | #Godavari', 'Akash Ravuru', 27), ('Appudo Ippudo | Bommarillu | Akash Ravuru | Cover | DSP | Siddharth | Genelia', 'Akash Ravuru', 26), ('Kesariya x Kumkumala | Bramhastra | Arijit Singh  | Sid Sriram | Pritam | Cover | Akash Ravuru', 'Akash Ravuru', 20), ('Kannulo Nee Roopame - Ninne Pelladtha | Cover | Akash Ravuru', 'Akash Ravuru', 20), ('#Sirivennela  | Shyam Singha Roy | Cover | #Nani #SaiPallavi | Akash Ravur